In [0]:
spark.conf.set(
    "fs.azure.account.key.ecommercedeproj1998.dfs.core.windows.net",
    ""
)

In [0]:
from pyspark.sql.functions import col, lit, current_date

dim_customer_existing = spark.read.format("delta").load(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/dim_customer"
).withColumn(
    "CustomerID",
    col("CustomerID").cast("long")
)

dim_customer_existing.printSchema()
dim_customer_existing.show(5)

In [0]:
dim_customer_scd2 = dim_customer_existing \
    .withColumn("StartDate", lit("2010-01-01").cast("date")) \
    .withColumn("EndDate", lit("9999-12-31").cast("date")) \
    .withColumn("IsCurrent", lit(True))
     dim_customer_scd2.show(5)

In [0]:
dim_customer_scd2.printSchema()

print(dim_customer_scd2.dtypes)

print(dim_customer_scd2.columns)



dim_customer_scd2.show(5)

dbutils.fs.rm(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/dim_customer_scd2",
    True
)

dim_customer_scd2.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@ecommercedeproj1998.dfs.core.windows.net/test_scd2")

print("Write Successful!")

In [0]:
from pyspark.sql.functions import col, when# Close out the old record (set EndDate and IsCurrent=False)
dim_customer_updated = dim_customer_scd2.withColumn(
    "EndDate", when(col("CustomerID") == 17850, lit("2025-12-31").cast("date")).otherwise(col("EndDate"))
).withColumn(
    "IsCurrent", when(col("CustomerID") == 17850, lit(False)).otherwise(col("IsCurrent"))
)

# Create a new row for that customer with new info
from pyspark.sql import Row
new_record = spark.createDataFrame([
    Row(CustomerID=17850, Country="Germany", StartDate="2026-01-01", 
        EndDate="9999-12-31", IsCurrent=True)
])

#  Union old (updated) + new record
dim_customer_final = dim_customer_updated.unionByName(new_record)
dim_customer_final.filter(col("CustomerID") == 17850).show()

In [0]:
dim_customer_existing.printSchema()
new_record.printSchema()

In [0]:
dim_customer_final.write.format("delta").mode("overwrite").save(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/dim_customer_scd2"
)
print("Final SCD2 table written successfully!")